In [2]:
# CELDA 1 — Setup e imports
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_experimental.agents import create_pandas_dataframe_agent

df = pd.read_csv("hotel_bookings.csv")

In [55]:
df = df[['hotel', 'is_canceled', 'adr', 
         'reservation_status', 'arrival_date_month',
         'arrival_date_year', 'customer_type',
         'market_segment']]

In [38]:
llm = ChatOllama(model="qwen2.5-coder:7b", temperature=0)

INSTRUCCIONES_FIJAS = """
Eres un analista de datos con pandas.
REGLAS CRÍTICAS:
- Action SIEMPRE debe ser: python_repl_ast
- NUNCA uses otro Action
- NUNCA escribas texto como Action
- Ejecuta TODO en python_repl_ast
- Una sola línea de código por Action Input

Ejemplo correcto:
Thought: necesito contar filas
Action: python_repl_ast
Action Input: len(df)
Observation: 119390
Final Answer: 119390
"""

In [39]:
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=INSTRUCCIONES_FIJAS,
    agent_type="zero-shot-react-description",
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=5
)

In [40]:
# CELDA 2
agent.invoke("¿Cuántas reservas hay en total y qué porcentaje fue cancelado?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito contar el número total de reservas y calcular el porcentaje de las que fueron canceladas.
Action: python_repl_ast
Action Input: len(df)119390Necesito calcular el número de reservas canceladas y no canceladas para poder determinar el porcentaje de cancelaciones.
Action: python_repl_ast
Action Input: df['is_canceled'].value_counts()is_canceled
0    75166
1    44224
Name: count, dtype: int64Necesito calcular el porcentaje de reservas canceladas.
Action: python_repl_ast
Action Input: (df['is_canceled'] == 1).mean() * 10037.041628277075134Final Answer: El número total de reservas es 119,390 y el porcentaje de reservas canceladas es aproximadamente 37.04%.

> Finished chain.


'El número total de reservas es 119,390 y el porcentaje de reservas canceladas es aproximadamente 37.04%.'

In [41]:
# CELDA 3
agent.invoke("¿Cuál es el ADR promedio por tipo de hotel?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito calcular el ADR promedio agrupado por el tipo de hotel.
Action: python_repl_ast
Action Input: df.groupby('hotel')['adr'].mean()hotel
City Hotel      105.304465
Resort Hotel     94.952930
Name: adr, dtype: float64Final Answer: El ADR promedio por tipo de hotel es:
City Hotel: 105.304465
Resort Hotel: 94.952930

> Finished chain.


'El ADR promedio por tipo de hotel es:\nCity Hotel: 105.304465\nResort Hotel: 94.952930'

In [42]:
# CELDA 4
agent.invoke("¿Qué mes tuvo más cancelaciones?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito contar las cancelaciones por mes.
Action: python_repl_ast
Action Input: df[df['is_canceled'] == 1]['arrival_date_month'].value_counts()arrival_date_month
August       5239
July         4742
May          4677
June         4535
April        4524
October      4246
September    4116
March        3149
February     2696
December     2371
November     2122
January      1807
Name: count, dtype: int64Final Answer: El mes con más cancelaciones fue agosto con 5239 cancelaciones.

> Finished chain.


'El mes con más cancelaciones fue agosto con 5239 cancelaciones.'

In [44]:
# CELDA 5
# CELDA 5
agent.invoke("¿Cuántos registros tienen reservation_status igual a 'No-Show' y cuánto ingreso perdido representan sumando su ADR?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito filtrar los registros donde reservation_status es 'No-Show' y luego calcular la suma de sus ADR.
Action: python_repl_ast
Action Input: df[df['reservation_status'] == 'No-Show'].shape[0]1207Necesito calcular la suma de los ADR para los registros donde reservation_status es 'No-Show'.
Action: python_repl_ast
Action Input: df[df['reservation_status'] == 'No-Show']['adr'].sum()116328.8Final Answer: Hay 1207 registros con reservation_status igual a 'No-Show' y el ingreso perdido representado por la suma de sus ADR es 116328.8.

> Finished chain.


"Hay 1207 registros con reservation_status igual a 'No-Show' y el ingreso perdido representado por la suma de sus ADR es 116328.8."

In [57]:
df.head()

,hotel,is_canceled,adr,reservation_status,arrival_date_month,arrival_date_year,customer_type,market_segment
0,Resort Hotel,0,0.0,Check-Out,July,2015,Transient,Direct
1,Resort Hotel,0,0.0,Check-Out,July,2015,Transient,Direct
2,Resort Hotel,0,75.0,Check-Out,July,2015,Transient,Direct
3,Resort Hotel,0,75.0,Check-Out,July,2015,Transient,Corporate
4,Resort Hotel,0,98.0,Check-Out,July,2015,Transient,Online TA


In [58]:
agent.invoke("¿Qué customer_type genera el mayor promedio de adr?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito calcular el promedio de ADR por cada tipo de cliente y luego identificar el que tiene el mayor promedio.
Action: python_repl_ast
Action Input: df.groupby('customer_type')['adr'].mean().sort_values(ascending=False).head(1)customer_type
Transient    107.013621
Name: adr, dtype: float64Final Answer: Transient

> Finished chain.


'Transient'

In [59]:
agent.invoke("¿Qué combinación de hotel y market_segment tiene el adr promedio más alto?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito calcular el adr promedio por cada combinación de hotel y market_segment.
Action: python_repl_ast
Action Input: df.groupby(['hotel', 'market_segment'])['adr'].mean().reset_index()           hotel market_segment         adr
0     City Hotel       Aviation  100.142110
1     City Hotel  Complementary    2.600424
2     City Hotel      Corporate   83.119980
3     City Hotel         Direct  119.479682
4     City Hotel         Groups   84.921885
5     City Hotel  Offline TA/TO   93.017660
6     City Hotel      Online TA  118.919533
7     City Hotel      Undefined   15.000000
8   Resort Hotel  Complementary    3.657413
9   Resort Hotel      Corporate   51.563183
10  Resort Hotel         Direct  111.670840
11  Resort Hotel         Groups   66.446964
12  Resort Hotel  Offline TA/TO   74.662571
13  Resort Hotel      Online TA  113.432480Observation:            hotel market_segment         adr
0     City Hotel       Aviation  100.142110
1   

'City Hotel - Direct'

In [61]:
# CELDA A
agent.invoke("¿Cuál es el market_segment con más registros con is_canceled igual a 1?")['output']



> Entering new AgentExecutor chain...
Thought: Necesito filtrar los registros donde `is_canceled` es igual a 1 y luego agrupar por `market_segment` para contar los registros en cada grupo.
Action: python_repl_ast
Action Input: df[df['is_canceled'] == 1].groupby('market_segment').size().idxmax()Online TAFinal Answer: Online TA

> Finished chain.


'Online TA'